# Contract Intelligence Multi-Agent System
## Assignment Notebook - 8-Week Capstone Project

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Welcome to Your Capstone Assignment!

This notebook is your guided journey to building a **production-grade multi-agent contract intelligence system**. Unlike the master solution, YOU will implement the core functionality.

### How This Assignment Works

1. **Scaffolding Provided**: Class structures, function signatures, and imports are given
2. **TODO Markers**: Look for `# TODO:` comments - these are YOUR tasks
3. **Hints**: Each section has hints to guide you (but not give away the answer)
4. **Validation Cells**: Run these to check if your implementation is correct
5. **Expected Output**: Sample outputs show what success looks like

### Difficulty Progression

| Week | Difficulty | Focus |
|------|-----------|-------|
| 1 | Easy | Environment setup (mostly provided) |
| 2 | Easy-Medium | Document processing methods |
| 3 | Medium | Vector store search functions |
| 4 | Medium-Hard | Implement agents from scratch |
| 5 | Hard | Multi-agent orchestration |
| 6 | Medium | Build knowledge graph |
| 7 | Medium | Add observability metrics |
| 8 | Hard | Integrate everything |

### Grading Rubric

- **Week 1-2**: 15% (Foundation)
- **Week 3-4**: 30% (Core Components)  
- **Week 5-6**: 30% (Advanced Features)
- **Week 7-8**: 25% (Production Polish)

Let's begin!

---

# WEEK 1: Environment & Foundations

**Difficulty: Easy** - Most code is provided. Focus on understanding.

---

## Learning Objectives

By the end of Week 1, you will:
- [ ] Set up all required packages
- [ ] Configure API keys securely
- [ ] Initialize Langfuse for observability
- [ ] Create traced wrapper functions
- [ ] Explore the contract data structure

---

## 1.1 Package Installation

This section is **provided** - just run the cell to install packages.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION (PROVIDED)
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## 1.2 Environment Configuration

This section is **provided** - handles environment detection and imports.

In [ ]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Python Version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION (PROVIDED)
# ============================================================================

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "contract_intelligence"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

PROJECT_NAME = "contract-intelligence-system"
DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

## 1.3 Langfuse Initialization

**YOUR FIRST TODO!** Initialize the Langfuse client.

### Hints:
- Import `Langfuse` from `langfuse`
- Import `observe` and `langfuse_context` from `langfuse.decorators`
- Use environment variables for credentials
- Call `auth_check()` to verify connection

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================
# TODO: Import necessary Langfuse modules

from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI

# TODO: Initialize the Langfuse client
# Hint: Use os.environ.get() to retrieve keys
# The constructor takes: secret_key, public_key, host

langfuse = None  # TODO: Replace with actual initialization

# YOUR CODE HERE:
# langfuse = Langfuse(
#     secret_key=???,
#     public_key=???,
#     host=???
# )

# TODO: Verify connection
# Hint: Use langfuse.auth_check() in a try/except block

# YOUR CODE HERE:


# TODO: Create a unique session ID
# Hint: Use datetime.now().strftime() to create a unique identifier
# Format: "contract-intel-YYYYMMDD-HHMMSS"

SESSION_ID = None  # TODO: Replace with actual session ID

# YOUR CODE HERE:


print(f"Session ID: {SESSION_ID}")

### Expected Output for 1.3:
```
Langfuse connection verified!
Session ID: contract-intel-20250127-143052
```

## 1.4 Traced Wrapper Functions

**TODO:** Implement traced versions of OpenAI calls.

### Hints:
- Create a trace with `langfuse.trace()`
- Create a generation span with `trace.generation()`
- Use `openai_client.embeddings.create()` for embeddings
- End the generation with usage stats

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED EMBEDDING FUNCTION
# ============================================================================

# Initialize OpenAI client (PROVIDED)
openai_client = OpenAI()

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Args:
        text: The text to embed
        trace_name: Identifier for this trace

    Returns:
        List of floats (embedding vector)

    TODO: Implement this function

    Steps:
    1. Create a trace with langfuse.trace()
    2. Create a generation span for the embedding call
    3. Call openai_client.embeddings.create()
    4. End the generation with output metadata
    5. Return the embedding
    """

    # TODO: Create trace
    # Hint: trace = langfuse.trace(name=..., session_id=SESSION_ID, metadata={...})

    # YOUR CODE HERE:
    trace = None


    # TODO: Create generation span
    # Hint: generation = trace.generation(name="openai-embedding", model="text-embedding-3-small", input=...)

    # YOUR CODE HERE:
    generation = None


    # TODO: Make API call
    # Hint: response = openai_client.embeddings.create(model="text-embedding-3-small", input=text)

    # YOUR CODE HERE:
    embedding = None


    # TODO: End generation with metadata
    # Hint: generation.end(output={...}, usage={...})

    # YOUR CODE HERE:


    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(prompt: str, trace_name: str = "completion") -> str:
    """
    Generate completion with full Langfuse tracing.

    Args:
        prompt: The prompt to complete
        trace_name: Identifier for this trace

    Returns:
        Generated text

    TODO: Implement this function (similar pattern to traced_embedding)
    """

    # TODO: Create trace
    # YOUR CODE HERE:
    trace = None


    # TODO: Create generation span
    # YOUR CODE HERE:
    generation = None


    # TODO: Make API call using openai_client.chat.completions.create()
    # Hint: Use model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}]

    # YOUR CODE HERE:
    response_text = None


    # TODO: End generation with usage stats
    # YOUR CODE HERE:


    return response_text

print("traced_completion() function defined")

## 1.5 Contract Document Taxonomy

**PROVIDED** - Study this structure carefully, you'll need it later.

In [ ]:
# ============================================================================
# WEEK 1.5: CONTRACT TAXONOMY (PROVIDED)
# ============================================================================

CONTRACT_CATEGORIES = {
    'master_agreements': {
        'path': 'Master level agreements',
        'description': 'Foundation contracts establishing overall relationship',
        'document_types': ['MSA', 'NDA', 'Rate Cards'],
        'risk_focus': ['legal', 'compliance'],
    },
    'transaction_contracts': {
        'path': 'Transaction level contract',
        'description': 'Project-specific agreements under master agreements',
        'document_types': ['SOW', 'Work Orders', 'Renewals', 'Amendments'],
        'risk_focus': ['operational', 'financial'],
    },
    'commercial_docs': {
        'path': 'Commercial docs',
        'description': 'Pricing and commercial terms documentation',
        'document_types': ['Pricing Tables', 'Rate Schedules'],
        'risk_focus': ['financial'],
    },
    'financial_billing': {
        'path': 'Financial and Billing docs',
        'description': 'Invoices, payment records, and billing schedules',
        'document_types': ['Invoices', 'Credit Notes', 'Payment Schedules'],
        'risk_focus': ['financial'],
    },
    'operational_docs': {
        'path': 'Operational service delivery docs',
        'description': 'Service delivery and operational documentation',
        'document_types': ['Service Reports', 'SLA Reports'],
        'risk_focus': ['operational'],
    },
    'compliance_docs': {
        'path': 'Compliance and policy documents',
        'description': 'Security, compliance, and policy documentation',
        'document_types': ['InfoSec Controls', 'Vendor Policies'],
        'risk_focus': ['compliance'],
    },
    'legal_support': {
        'path': 'Legal support documents',
        'description': 'Legal correspondence and supporting documentation',
        'document_types': ['Legal Memos', 'Correspondence'],
        'risk_focus': ['legal'],
    },
    'relationship_governance': {
        'path': 'Relationship and governance docs',
        'description': 'Governance frameworks and relationship management',
        'document_types': ['Governance Frameworks', 'Escalation Matrices'],
        'risk_focus': ['operational', 'compliance'],
    }
}

print(f"Contract categories defined: {len(CONTRACT_CATEGORIES)}")
for cat, info in CONTRACT_CATEGORIES.items():
    print(f"  - {cat}: {info['description'][:50]}...")

## 1.6 Data Discovery

**TODO:** Implement the data discovery function.

### Hints:
- Use `Path.rglob('*')` to recursively find files
- Check file suffixes with `item.suffix.lower()`
- Create spans for each category scan

In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================

def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all contract documents organized by category.

    Args:
        data_dir: Path to the root data directory

    Returns:
        Dictionary mapping categories to lists of file paths

    TODO: Implement this function

    Steps:
    1. Create a trace for data discovery
    2. Initialize discovered dict with empty lists for each category
    3. For each category, scan the appropriate subdirectory
    4. Find all .docx, .pdf, .xlsx files
    5. Log results to Langfuse
    """

    # TODO: Create trace for data discovery
    # YOUR CODE HERE:
    trace = None


    # TODO: Initialize discovered dictionary
    # Hint: discovered = {cat: [] for cat in CONTRACT_CATEGORIES.keys()}

    # YOUR CODE HERE:
    discovered = {}


    # TODO: Check if data directory exists
    # YOUR CODE HERE:


    # TODO: Scan each category
    # Hint: Loop through CONTRACT_CATEGORIES, build path, use rglob
    for category, info in CONTRACT_CATEGORIES.items():
        # YOUR CODE HERE:
        pass


    # TODO: Calculate and log totals
    # YOUR CODE HERE:


    return discovered

# Set up data path
if IN_COLAB:
    DATA_DIR = Path('/content/data')
else:
    DATA_DIR = Path('../data')

# Run discovery
contract_files = discover_contract_data(DATA_DIR)

print("\nDiscovery Summary:")
total = 0
for cat, files in contract_files.items():
    if files:
        print(f"  {cat}: {len(files)} files")
        total += len(files)
print(f"\nTotal files discovered: {total}")

## Checkpoint: Week 1 Verification

Run this cell to verify your Week 1 implementation.

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Langfuse initialized
try:
    checks.append(("Langfuse initialized", langfuse is not None))
except:
    checks.append(("Langfuse initialized", False))

# Check 2: Session ID created
try:
    checks.append(("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 10))
except:
    checks.append(("Session ID created", False))

# Check 3: OpenAI client ready
try:
    checks.append(("OpenAI client ready", openai_client is not None))
except:
    checks.append(("OpenAI client ready", False))

# Check 4: traced_embedding works
try:
    test_emb = traced_embedding("test", "test-validation")
    checks.append(("traced_embedding() works", test_emb is not None and len(test_emb) == 1536))
except Exception as e:
    checks.append(("traced_embedding() works", False))
    print(f"  Error: {e}")

# Check 5: traced_completion works
try:
    test_comp = traced_completion("Say 'hello'", "test-validation")
    checks.append(("traced_completion() works", test_comp is not None and len(test_comp) > 0))
except Exception as e:
    checks.append(("traced_completion() works", False))
    print(f"  Error: {e}")

# Check 6: Contract taxonomy
checks.append(("Contract taxonomy defined", len(CONTRACT_CATEGORIES) == 8))

# Check 7: Data discovery
try:
    checks.append(("Data discovery function works", callable(discover_contract_data)))
except:
    checks.append(("Data discovery function works", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")
    print("\nHint: Check the error messages and compare with expected output.")

---

# WEEK 2: Document Processing & EDA

**Difficulty: Easy-Medium** - Implement document parsing methods.

---

## Learning Objectives

By the end of Week 2, you will:
- [ ] Load and parse DOCX contract files
- [ ] Extract text from tables
- [ ] Identify document sections
- [ ] Perform exploratory data analysis
- [ ] Extract contract entities

---

## 2.1 Document Processor Class

**TODO:** Implement the core document processing methods.

### Hints:
- Use `python-docx` library (already imported as `Document`)
- Paragraphs are in `doc.paragraphs`
- Tables are in `doc.tables`
- Use regex patterns to identify section headers

In [ ]:
# ============================================================================
# WEEK 2.1: DOCUMENT PROCESSOR CLASS
# ============================================================================

from docx import Document as DocxDocument
import re
import hashlib

class ContractDocumentProcessor:
    """
    Production-grade document processor for contract analysis.

    YOU MUST IMPLEMENT:
    - load_docx(): Load and parse a DOCX file
    - extract_tables(): Extract text from tables
    - extract_sections(): Identify document sections
    - process_category(): Process all documents in a category
    """

    def __init__(self):
        self.supported_formats = ['.docx']
        self.processed_docs = []

        # Section header patterns (PROVIDED)
        self.section_patterns = [
            r'^(\d+\.\s+[A-Z][A-Z\s]+)$',      # "1. DEFINITIONS"
            r'^(\d+\.\d+\s+.+)$',              # "1.1 Term"
            r'^(ARTICLE\s+[IVXLCDM]+)',          # "ARTICLE I"
            r'^(SECTION\s+\d+)',                # "SECTION 1"
            r'^(Schedule\s+[A-Z0-9]+)',          # "Schedule A"
            r'^(Exhibit\s+[A-Z0-9]+)',           # "Exhibit A"
        ]

    def load_docx(self, file_path: Path, trace_parent=None) -> Optional[Dict[str, Any]]:
        """
        Load and parse a DOCX contract file.

        Args:
            file_path: Path to the DOCX file
            trace_parent: Parent trace for nested spans

        Returns:
            Dictionary with parsed document data, or None on error

        TODO: Implement this method

        Steps:
        1. Create a span if trace_parent exists
        2. Open the DOCX file using DocxDocument(file_path)
        3. Extract all paragraph text
        4. Extract table text using extract_tables()
        5. Generate a unique doc_id
        6. Return dictionary with: doc_id, filename, full_text, table_text, sections
        """

        # TODO: Create span for tracing (if parent exists)
        span = trace_parent.span(name=f"load-{file_path.name}") if trace_parent else None

        try:
            # TODO: Load the DOCX file
            # Hint: doc = DocxDocument(file_path)

            # YOUR CODE HERE:
            doc = None


            # TODO: Extract paragraph text
            # Hint: Loop through doc.paragraphs and join their text

            # YOUR CODE HERE:
            full_text = ""


            # TODO: Extract table text
            # Hint: Use self.extract_tables(doc)

            # YOUR CODE HERE:
            table_text = ""


            # TODO: Generate unique doc_id
            # Hint: Use hashlib.md5 on filename

            # YOUR CODE HERE:
            doc_id = ""


            # TODO: Extract sections
            # Hint: Use self.extract_sections(full_text)

            # YOUR CODE HERE:
            sections = []


            # End span if exists
            if span:
                span.end(output={"status": "success", "text_length": len(full_text)})

            return {
                'doc_id': doc_id,
                'filename': file_path.name,
                'file_path': str(file_path),
                'full_text': full_text,
                'table_text': table_text,
                'sections': sections,
                'word_count': len(full_text.split())
            }

        except Exception as e:
            if span:
                span.end(output={"status": "error", "error": str(e)})
            print(f"Error loading {file_path}: {e}")
            return None

    def extract_tables(self, doc) -> str:
        """
        Extract text from all tables in a document.

        Args:
            doc: A python-docx Document object

        Returns:
            String containing all table text

        TODO: Implement this method

        Hints:
        - Tables are in doc.tables
        - Each table has .rows
        - Each row has .cells
        - Each cell has .text
        """

        # YOUR CODE HERE:
        table_texts = []

        # TODO: Loop through doc.tables
        # TODO: For each table, loop through rows
        # TODO: For each row, extract cell text
        # TODO: Join into formatted string

        return "\n".join(table_texts)

    def extract_sections(self, text: str) -> List[Dict[str, str]]:
        """
        Extract sections from document text using regex patterns.

        Args:
            text: Full document text

        Returns:
            List of dictionaries with section info

        TODO: Implement this method

        Hints:
        - Split text into lines
        - Check each line against self.section_patterns
        - Track section headers and their positions
        """

        # YOUR CODE HERE:
        sections = []

        # TODO: Split text into lines
        # TODO: For each line, check against section patterns
        # TODO: If match found, record section header

        return sections

print("ContractDocumentProcessor class defined")
print("TODO: Implement load_docx(), extract_tables(), extract_sections()")

### Expected Output for load_docx():
```python
{
    'doc_id': 'ABC123...',
    'filename': 'sample_msa.docx',
    'full_text': 'MASTER SERVICE AGREEMENT...',
    'table_text': 'Service | Price | Term...',
    'sections': [{'header': '1. DEFINITIONS', 'position': 0}, ...],
    'word_count': 5432
}
```

In [ ]:
# ============================================================================
# WEEK 2.1: PROCESS CATEGORY METHOD
# ============================================================================

def process_category(self, files: List[Path], category: str) -> List[Dict]:
    """
    Process all documents in a category with tracing.

    Args:
        files: List of file paths
        category: Category name

    Returns:
        List of processed document dictionaries

    TODO: Implement this method
    """

    # TODO: Create trace for this category
    trace = langfuse.trace(
        name=f"process-{category}",
        session_id=SESSION_ID,
        input={"category": category, "file_count": len(files)}
    )

    processed = []

    # TODO: Loop through files and process each one
    for file_path in files:
        # TODO: Check if file format is supported
        # TODO: Call load_docx with trace as parent
        # TODO: Add category to result
        # TODO: Append to processed list

        # YOUR CODE HERE:
        pass

    # Update trace with results
    trace.update(output={"processed_count": len(processed)})
    self.processed_docs.extend(processed)

    return processed

# Add method to class
ContractDocumentProcessor.process_category = process_category

print("process_category() method added")

## 2.2 Process All Documents

Run your document processor on all discovered files.

In [ ]:
# ============================================================================
# WEEK 2.2: PROCESS ALL DOCUMENTS
# ============================================================================

# Initialize processor
doc_processor = ContractDocumentProcessor()

print("PROCESSING CONTRACT DOCUMENTS")
print("=" * 60)

# Process all categories
all_contract_docs = {}
for category, files in contract_files.items():
    if files:
        all_contract_docs[category] = doc_processor.process_category(files, category)
        print(f"{category}: {len(all_contract_docs[category])} documents processed")

# Flatten all documents
all_docs_flat = []
for docs in all_contract_docs.values():
    all_docs_flat.extend(docs)

print(f"\n" + "=" * 60)
print(f"TOTAL DOCUMENTS PROCESSED: {len(all_docs_flat)}")

# If no real documents, create sample data
if not all_docs_flat:
    print("\nNo documents found - creating sample data for demonstration...")
    sample_contract_text = """
MASTER SERVICE AGREEMENT

This Master Service Agreement ("Agreement") is entered into as of January 1, 2024
between TechCorp Inc. ("Client") and ServicePro LLC ("Provider").

1. DEFINITIONS
   "Services" means the professional services described in any Statement of Work.
   "Confidential Information" means any non-public information disclosed by either party.

2. SERVICES AND DELIVERABLES
   Provider shall perform the Services described in each Statement of Work.
   All deliverables shall meet the specifications set forth in the applicable SOW.

3. PAYMENT TERMS
   Client shall pay Provider within Net 30 days of invoice receipt.
   Late payments shall accrue interest at 1.5% per month.
   Total contract value: $500,000 over 24 months.

4. LIABILITY
   Provider's total liability shall not exceed the fees paid in the prior 12 months.
   Neither party shall be liable for consequential, indirect, or punitive damages.

5. INDEMNIFICATION
   Each party shall indemnify the other against third-party claims arising from
   their negligence or willful misconduct.

6. TERMINATION
   Either party may terminate with 30 days written notice.
   Client may terminate for cause immediately upon material breach.
"""

    all_docs_flat = [{
        'doc_id': 'SAMPLE-001',
        'filename': 'sample_msa.docx',
        'category': 'master_agreements',
        'full_text': sample_contract_text,
        'table_text': '',
        'sections': [{'header': '1. DEFINITIONS', 'position': 0}],
        'word_count': len(sample_contract_text.split())
    }]

    print(f"Sample document created with {all_docs_flat[0]['word_count']} words")

langfuse.flush()

## 2.3 Exploratory Data Analysis (EDA)

**TODO:** Create visualizations to understand the contract corpus.

### Hints:
- Use matplotlib/seaborn for visualizations
- Create charts for: category distribution, word counts, section counts

In [ ]:
# ============================================================================
# WEEK 2.3: EXPLORATORY DATA ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# TODO: Create a DataFrame from all_docs_flat
# Hint: df = pd.DataFrame(all_docs_flat)

# YOUR CODE HERE:
df = None


# TODO: Create visualization 1 - Category Distribution
# Hint: Use df['category'].value_counts() and plt.bar()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# YOUR CODE HERE for axes[0, 0]:
# Category distribution bar chart


# TODO: Create visualization 2 - Word Count Distribution
# Hint: Use df['word_count'] and plt.hist()

# YOUR CODE HERE for axes[0, 1]:
# Word count histogram


# TODO: Create visualization 3 - Word Count by Category
# Hint: Use seaborn boxplot

# YOUR CODE HERE for axes[1, 0]:
# Box plot of word counts by category


# TODO: Create visualization 4 - Summary Statistics
# Hint: Create a table or text summary

# YOUR CODE HERE for axes[1, 1]:
# Summary statistics display


plt.tight_layout()
plt.savefig('contract_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEDA visualizations saved to contract_eda.png")

## 2.4 Contract Entity Extraction

**TODO:** Implement entity extraction using regex patterns.

### Hints:
- Use `re.findall()` for pattern matching
- Common entities: parties, dates, monetary values, percentages
- Risk indicators: liability terms, termination clauses

In [ ]:
# ============================================================================
# WEEK 2.4: CONTRACT ENTITY EXTRACTOR
# ============================================================================

class ContractEntityExtractor:
    """
    Extract contract-specific entities using pattern matching.

    TODO: Implement extract_entities() and assess_risk_indicators()
    """

    def __init__(self):
        # Extraction patterns (PROVIDED)
        self.patterns = {
            'parties': r'(?:between|by and between)\s+([A-Z][A-Za-z\s]+(?:Inc\.|LLC|Ltd\.|Corp\.)?)',
            'effective_date': r'(?:effective\s+(?:as\s+of\s+)?(?:date)?:?\s*)(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|[A-Z][a-z]+\s+\d{1,2},?\s+\d{4})',
            'monetary_values': r'\$([\d,]+(?:\.\d{2})?)',
            'percentages': r'(\d+(?:\.\d+)?\s*%)',
            'durations': r'(\d+)\s*(?:years?|months?|days?)',
            'notice_periods': r'(?:notice\s+(?:period)?\s*(?:of)?\s*)(\d+)\s*(?:days?)',
        }

        # Risk indicators (PROVIDED)
        self.risk_indicators = {
            'high_risk': [
                'unlimited liability', 'indemnify', 'consequential damages',
                'punitive damages', 'sole discretion', 'automatic renewal'
            ],
            'medium_risk': [
                'liability cap', 'limitation of liability', 'force majeure',
                'material breach', 'termination for convenience'
            ],
            'low_risk': [
                'mutual indemnification', 'standard terms', 'annual review'
            ]
        }

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """
        Extract all entities from contract text.

        Args:
            text: Contract text

        Returns:
            Dictionary mapping entity types to found values

        TODO: Implement this method

        Hints:
        - Loop through self.patterns
        - Use re.findall(pattern, text, re.IGNORECASE)
        - Store results in dictionary
        """

        # YOUR CODE HERE:
        entities = {}

        # TODO: For each pattern, find all matches


        return entities

    def assess_risk_indicators(self, text: str) -> Dict[str, List[str]]:
        """
        Identify risk indicators in contract text.

        Args:
            text: Contract text

        Returns:
            Dictionary with found risk indicators by level

        TODO: Implement this method

        Hints:
        - Loop through self.risk_indicators
        - Check if each indicator phrase is in text.lower()
        - Track which indicators were found
        """

        # YOUR CODE HERE:
        found_risks = {'high_risk': [], 'medium_risk': [], 'low_risk': []}

        # TODO: Check each risk indicator


        return found_risks

# Initialize extractor
entity_extractor = ContractEntityExtractor()

print("ContractEntityExtractor class defined")
print("TODO: Implement extract_entities() and assess_risk_indicators()")

In [ ]:
# ============================================================================
# TEST ENTITY EXTRACTION
# ============================================================================

# Test on sample document
if all_docs_flat:
    test_doc = all_docs_flat[0]
    print(f"Testing entity extraction on: {test_doc['filename']}")
    print("=" * 60)

    # Extract entities
    entities = entity_extractor.extract_entities(test_doc['full_text'])
    print("\nExtracted Entities:")
    for entity_type, values in entities.items():
        if values:
            print(f"  {entity_type}: {values[:3]}...")  # Show first 3

    # Assess risk
    risks = entity_extractor.assess_risk_indicators(test_doc['full_text'])
    print("\nRisk Indicators Found:")
    for level, indicators in risks.items():
        if indicators:
            print(f"  {level}: {indicators}")

## Checkpoint: Week 2 Verification

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 2 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Document processor exists
checks.append(("DocumentProcessor initialized", doc_processor is not None))

# Check 2: load_docx method works
try:
    # Test with sample text (doesn't need real file)
    test_result = doc_processor.load_docx is not None
    checks.append(("load_docx() method exists", test_result))
except:
    checks.append(("load_docx() method exists", False))

# Check 3: Documents processed
checks.append(("Documents processed", len(all_docs_flat) > 0))

# Check 4: Entity extractor works
try:
    test_entities = entity_extractor.extract_entities("Contract value is $100,000")
    has_monetary = 'monetary_values' in test_entities
    checks.append(("Entity extraction works", has_monetary))
except Exception as e:
    checks.append(("Entity extraction works", False))
    print(f"  Error: {e}")

# Check 5: Risk assessment works
try:
    test_risks = entity_extractor.assess_risk_indicators("unlimited liability clause")
    has_high = len(test_risks.get('high_risk', [])) > 0
    checks.append(("Risk assessment works", has_high))
except Exception as e:
    checks.append(("Risk assessment works", False))
    print(f"  Error: {e}")

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 2 CHECKPOINTS PASSED! Ready for Week 3.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 3: Vector Store & Embeddings

**Difficulty: Medium** - Implement semantic search functionality.

---

## Learning Objectives

By the end of Week 3, you will:
- [ ] Initialize ChromaDB for vector storage
- [ ] Implement document indexing with embeddings
- [ ] Create semantic search functions
- [ ] Understand similarity scoring

---

## 3.1 ChromaDB Vector Store Class

**TODO:** Implement the vector store with search capabilities.

### Hints:
- Use `chromadb.PersistentClient` for storage
- Create collections with `client.get_or_create_collection()`
- Use your `traced_embedding()` function for embeddings
- ChromaDB's `query()` method returns results with distances

In [ ]:
# ============================================================================
# WEEK 3.1: CHROMADB VECTOR STORE
# ============================================================================

import chromadb

class ContractVectorStore:
    """
    ChromaDB-based vector store with Langfuse observability.

    TODO: Implement these methods:
    - create_collection(): Create or get a collection
    - add_documents(): Index documents with embeddings
    - search(): Semantic search across documents
    - search_with_filter(): Search with metadata filters
    """

    def __init__(self, persist_directory: str = "./chroma_contracts_db"):
        """
        Initialize ChromaDB with persistent storage.

        TODO: Complete the initialization
        """

        # TODO: Create persistent client
        # Hint: self.client = chromadb.PersistentClient(path=persist_directory)

        # YOUR CODE HERE:
        self.client = None


        self.embedding_model = "text-embedding-3-small"
        self.collections = {}

        # Log initialization to Langfuse
        langfuse.trace(
            name="vectorstore-init",
            session_id=SESSION_ID,
            input={"persist_directory": persist_directory}
        )

        print(f"ChromaDB initialized at: {persist_directory}")

    def create_collection(self, name: str):
        """
        Create or get a collection.

        Args:
            name: Collection name

        Returns:
            ChromaDB collection

        TODO: Implement this method

        Hints:
        - Use self.client.get_or_create_collection(name=name)
        - Store in self.collections[name]
        """

        # YOUR CODE HERE:
        collection = None


        self.collections[name] = collection
        return collection

    def add_documents(self, collection_name: str, documents: List[Dict]):
        """
        Index documents into a collection.

        Args:
            collection_name: Target collection name
            documents: List of document dictionaries

        TODO: Implement this method

        Steps:
        1. Get or create collection
        2. For each document:
           a. Generate embedding using traced_embedding()
           b. Prepare metadata (filename, category, word_count)
           c. Add to collection with doc_id as ID
        3. Log progress

        Hints:
        - Use collection.add(ids=[...], embeddings=[...], documents=[...], metadatas=[...])
        - Truncate text before embedding (max ~8000 chars)
        """

        # Create trace for indexing
        trace = langfuse.trace(
            name=f"index-{collection_name}",
            session_id=SESSION_ID,
            input={"doc_count": len(documents)}
        )

        # Get or create collection
        collection = self.create_collection(collection_name)

        # TODO: Process and add documents
        ids = []
        embeddings = []
        docs = []
        metadatas = []

        for i, doc in enumerate(documents):
            # TODO: Generate embedding
            # Hint: Use traced_embedding(doc['full_text'][:8000], f"embed-{doc['doc_id']}")

            # YOUR CODE HERE:


            # TODO: Prepare metadata
            # YOUR CODE HERE:


            # Progress indicator
            if (i + 1) % 5 == 0:
                print(f"  Indexed {i + 1}/{len(documents)} documents...")

        # TODO: Add all to collection
        # Hint: collection.add(ids=ids, embeddings=embeddings, documents=docs, metadatas=metadatas)

        # YOUR CODE HERE:


        trace.update(output={"indexed": len(documents)})
        print(f"Indexed {len(documents)} documents into '{collection_name}'")

    def search(self, collection_name: str, query: str, n_results: int = 5) -> List[Dict]:
        """
        Semantic search across documents.

        Args:
            collection_name: Collection to search
            query: Search query text
            n_results: Number of results to return

        Returns:
            List of result dictionaries with document, metadata, and score

        TODO: Implement this method

        Steps:
        1. Generate embedding for query
        2. Query collection using query_embeddings
        3. Format results with scores

        Hints:
        - Use collection.query(query_embeddings=[query_embedding], n_results=n_results)
        - Results contain 'documents', 'metadatas', 'distances'
        - Convert distance to similarity: similarity = 1 - distance (for cosine)
        """

        # Create trace for search
        trace = langfuse.trace(
            name="semantic-search",
            session_id=SESSION_ID,
            input={"query": query, "collection": collection_name}
        )

        # TODO: Get collection
        # YOUR CODE HERE:
        collection = None


        # TODO: Generate query embedding
        # YOUR CODE HERE:
        query_embedding = None


        # TODO: Query collection
        # YOUR CODE HERE:
        results = None


        # TODO: Format results
        formatted = []
        # Hint: Loop through results and create dict with 'document', 'metadata', 'similarity'

        # YOUR CODE HERE:


        trace.update(output={"results_count": len(formatted)})
        return formatted

print("ContractVectorStore class defined")
print("TODO: Implement create_collection(), add_documents(), search()")

### Expected Output for search():
```python
[
    {
        'document': 'Provider shall perform the Services...',
        'metadata': {'filename': 'sample_msa.docx', 'category': 'master_agreements'},
        'similarity': 0.87
    },
    ...
]
```

In [ ]:
# ============================================================================
# WEEK 3.2: INITIALIZE AND INDEX DOCUMENTS
# ============================================================================

# Initialize vector store
vector_store = ContractVectorStore()

print("\n" + "=" * 60)
print("INDEXING CONTRACT DOCUMENTS")
print("=" * 60)

# Index all documents
if all_docs_flat:
    vector_store.add_documents("contracts_all", all_docs_flat)
else:
    print("No documents to index!")

langfuse.flush()
print("\nIndexing complete!")

## 3.3 Semantic Search Examples

Test your search implementation with various queries.

In [ ]:
# ============================================================================
# WEEK 3.3: SEMANTIC SEARCH EXAMPLES
# ============================================================================

print("SEMANTIC SEARCH EXAMPLES")
print("=" * 60)

# Test queries
test_queries = [
    "liability limitations and caps",
    "payment terms and invoicing",
    "termination for convenience",
    "intellectual property rights",
    "confidentiality obligations"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)

    results = vector_store.search("contracts_all", query, n_results=3)

    for i, result in enumerate(results, 1):
        print(f"  {i}. [{result.get('similarity', 0):.3f}] {result.get('metadata', {}).get('filename', 'unknown')}")
        # Show snippet
        doc_text = result.get('document', '')[:150]
        print(f"     '{doc_text}...'")

langfuse.flush()

## 3.4 Search with Metadata Filters (BONUS)

**BONUS TODO:** Implement filtered search.

### Hints:
- ChromaDB supports `where` parameter for filtering
- Filter format: `{"category": "master_agreements"}`

In [ ]:
# ============================================================================
# WEEK 3.4: FILTERED SEARCH (BONUS)
# ============================================================================

def search_with_filter(self, collection_name: str, query: str,
                       filter_dict: Dict = None, n_results: int = 5) -> List[Dict]:
    """
    Semantic search with metadata filtering.

    Args:
        collection_name: Collection to search
        query: Search query
        filter_dict: Metadata filter (e.g., {"category": "master_agreements"})
        n_results: Number of results

    Returns:
        Filtered search results

    BONUS TODO: Implement this method

    Hints:
    - Add `where=filter_dict` parameter to collection.query()
    """

    # YOUR CODE HERE:
    pass

# Add method to class
ContractVectorStore.search_with_filter = search_with_filter

print("search_with_filter() method added (BONUS)")

## Checkpoint: Week 3 Verification

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 3 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Vector store initialized
checks.append(("VectorStore initialized", vector_store is not None))

# Check 2: ChromaDB client created
try:
    checks.append(("ChromaDB client created", vector_store.client is not None))
except:
    checks.append(("ChromaDB client created", False))

# Check 3: Collection created
try:
    has_collection = "contracts_all" in vector_store.collections
    checks.append(("Collection created", has_collection))
except:
    checks.append(("Collection created", False))

# Check 4: Search returns results
try:
    results = vector_store.search("contracts_all", "liability", n_results=1)
    checks.append(("Search returns results", len(results) > 0))
except Exception as e:
    checks.append(("Search returns results", False))
    print(f"  Error: {e}")

# Check 5: Results have correct structure
try:
    if results:
        has_doc = 'document' in results[0]
        has_meta = 'metadata' in results[0]
        has_sim = 'similarity' in results[0]
        checks.append(("Result structure correct", has_doc and has_meta and has_sim))
    else:
        checks.append(("Result structure correct", False))
except:
    checks.append(("Result structure correct", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 3 CHECKPOINTS PASSED! Ready for Week 4.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")